# setup + imports

In [1]:
import os, json, random
import torch, pandas as pd
from PIL import Image
from torchvision import transforms
from transformers import CLIPModel, CLIPProcessor

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print(DEVICE)
NAME = "openai/clip-vit-large-patch14"

/Users/ecaterina/practice/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1030 20:00:20.780000 75903 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


mps


In [2]:
def load_split(split):
    items = []
    for id_ in sorted(os.listdir(split)):
        if not id_.isdigit():        # skip hidden folders like .ipynb_checkpoints
            continue
        with open(f"{split}/{id_}/metadata.json") as f:
            meta = json.load(f)
        meta["id"] = id_
        meta["image"] = f"{split}/{id_}/image.png"
        items.append(meta)
    return items

train = load_split("train")
test = load_split("test")
print(len(train), len(test))

196 100


In [3]:
train_data = load_split("train")
test_data = load_split("test")
random.seed(0)
random.shuffle(train)
fit, val = train[:156], train[156:]

# model + validare

In [4]:
model = CLIPModel.from_pretrained(NAME).to(DEVICE)
processor = CLIPProcessor.from_pretrained(NAME)

Loading weights: 100%|█████████████████████| 590/590 [00:00<00:00, 4759.82it/s]


In [5]:
def scores(img, words):
    pix = processor(images=img,return_tensors='pt')["pixel_values"].to(DEVICE)
    texts = [f"a photo of a {w.replace('_', ' ').lower()}" for w in words]
    tok = processor.tokenizer(texts, padding=True, return_tensors="pt").to(DEVICE)
    img_emb = model.get_image_features(pixel_values=pix).pooler_output
    txt_emb = model.get_text_features(**tok).pooler_output
    img_emb = img_emb / img_emb.norm(dim=-1, keepdim=True)
    txt_emb = txt_emb / txt_emb.norm(dim=-1, keepdim=True)
    return model.logit_scale.exp() * (img_emb @ txt_emb.T)[0]


def build_cats(items):
    cat = {}
    for it in items:
        for w, c in zip(it["correct_words"], it["categories_used"]):
            cat[w.lower()] = c
    return cat

cat = build_cats(train)
words = [w for it in test for w in it["word_choices"]]
print(sum(w.lower() in cat for w in words) / len(words))

def predict(it, cat):
    img = Image.open(it["image"]).convert("RGB")
    s = scores(img, it["word_choices"])
    s = s + scores(img.transpose(Image.FLIP_LEFT_RIGHT), it["word_choices"])
    ranked = [it["word_choices"][k] for k in s.argsort(descending=True).tolist()]

    top3, rest, used = [], [], set()
    for w in ranked:
        c = cat.get(w.lower())
        if len(top3) < 3 and (c is None or c not in used):
            top3.append(w)
            if c is not None:
                used.add(c)
        else:
            rest.append(w)
    return top3 + rest[:2]
    

0.583


In [6]:
def judge(items):
    model.eval()
    total = 0.0
    with torch.no_grad():
        for it in items:
            correct = {c.lower() for c in it["correct_words"]}
            for pos, w in enumerate(predict(it, build_cats(fit))):
                if w.lower() in correct:
                    total += 0.3 + (0.1 if pos < 3 else 0) 
    print(f"score {total:.1f} / {1.2 * len(items):.1f}")
    return total

judge(val) 

score 35.5 / 48.0


35.49999999999995

In [7]:
aug = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2)
])

opt = torch.optim.AdamW(model.parameters(), lr=1e-6)
loss_fn = torch.nn.BCEWithLogitsLoss()

best_score = judge(val)   # zero-shot score is the one to beat
best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

for epoch in range(5):
    model.train()
    total = 0.0
    random.shuffle(fit)

    for it in fit:
        img = aug(Image.open(it["image"])).convert('RGB')
        correct = {c.lower() for c in it["correct_words"]}
        target = torch.tensor([1.0 if w.lower() in correct else 0.0 for w in it["word_choices"]], device=DEVICE)
        loss = loss_fn(scores(img, it["word_choices"]), target)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item()
        
    score = judge(val)
    if score > best_score:
        best_score = score
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    print(epoch, round(total/len(fit), 3))

model.load_state_dict(best_state)   # back to the best version
print("best:", best_score)

score 35.5 / 48.0
score 38.0 / 48.0
0 0.79
score 39.2 / 48.0
1 0.205
score 38.9 / 48.0
2 0.121
score 39.7 / 48.0
3 0.063
score 40.0 / 48.0
4 0.035
best: 39.99999999999994


In [8]:
model.eval()
rows = []
with torch.no_grad():
    for it in test:
        rows.append([it["id"]] + predict(it, cat))
pd.DataFrame(rows, columns=["ID", "word1", "word2", "word3", "word4", "word5"]).to_csv("submission.csv", index=False)

